# GOLD 01. CREACION TABLAS

## 1. Configuración

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Gold_Creacion") \
    .getOrCreate()

FORMATO   = "PARQUET"                 # Fabric: "DELTA"
RUTA_GOLD = "gs://tfm_unir1/gold"     # Fabric: None
ESQUEMA   = "gold"                    # Fabric: "gold" o None

# Funciones auxiliares: devuelven el nombre completo de la tabla y su ubicación
def tabla(nombre):
    return f"{ESQUEMA}.{nombre}" if ESQUEMA else nombre

def ubicacion(nombre):
    return f"LOCATION '{RUTA_GOLD}/{nombre}'" if RUTA_GOLD else ""

26/10/24 20:38:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 2. Creación Esquema GOLD

In [4]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {ESQUEMA} LOCATION '{RUTA_GOLD}'")

spark.sql ("SHOW DATABASES").show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+
|namespace|
+---------+
|  default|
|     gold|
+---------+



## 3. Dimensiones

In [8]:
# dim_tiempo
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_tiempo")}(
        id_tiempo INT NOT NULL,
        fecha_hora TIMESTAMP NOT NULL,
        fecha DATE NOT NULL,
        anio INT,
        mes INT,
        dia INT,
        hora INT,
        es_fin_de_semana BOOLEAN,
        es_festivo BOOLEAN,
        nombre_festivo STRING,
        es_evento BOOLEAN,
        es_confinamiento BOOLEAN
    )
    USING {FORMATO}
    {ubicacion("dim_tiempo")}
    """)

DataFrame[]

In [9]:
spark.sql(f"DESCRIBE TABLE {tabla('dim_tiempo')}").show(truncate=False)

+----------------+---------+-------+
|col_name        |data_type|comment|
+----------------+---------+-------+
|id_tiempo       |int      |null   |
|fecha_hora      |timestamp|null   |
|fecha           |date     |null   |
|anio            |int      |null   |
|mes             |int      |null   |
|dia             |int      |null   |
|hora            |int      |null   |
|es_fin_de_semana|boolean  |null   |
|es_festivo      |boolean  |null   |
|nombre_festivo  |string   |null   |
|es_evento       |boolean  |null   |
|es_confinamiento|boolean  |null   |
+----------------+---------+-------+



In [10]:
# dim_estacion
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_estacion")}(
        id_estacion STRING NOT NULL,
        codigo_eea STRING,
        nombre_estacion STRING,
        tipo_estacion STRING,
        zona STRING,
        latitud DOUBLE,
        longitud DOUBLE,
        altitud DOUBLE,
        fecha_alta DATE,
        fecha_baja DATE,
        activa BOOLEAN
    )
    USING {FORMATO}
    {ubicacion("dim_estacion")}
    """)

26/10/24 20:58:43 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/dim_estacion was not found. Was it deleted very recently?


DataFrame[]

In [11]:
# dim_contaminante
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_contaminante")}(
        id_contaminante INT NOT NULL,
        nombre STRING,
        unidad STRING,
        indicador STRING,
        limite_ue_actual DOUBLE,
        objetivo_ue_2030 DOUBLE,
        guia_oms_2021 DOUBLE
    )
    USING {FORMATO}
    {ubicacion("dim_contaminante")}
    """)

26/10/24 21:00:31 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/dim_contaminante was not found. Was it deleted very recently?


DataFrame[]

## 4. Tablas de hechos

In [12]:
# fact_aire
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_aire")}(
        id_tiempo INT NOT NULL,
        id_estacion STRING NOT NULL,
        id_contaminante INT NOT NULL,
        concentracion DOUBLE
    )
    USING {FORMATO}
    {ubicacion("fact_aire")}
    """)

26/10/24 21:02:55 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/fact_aire was not found. Was it deleted very recently?


DataFrame[]

In [13]:
# fact_meteo
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_meteo")}(
        id_tiempo INT NOT NULL,
        id_estacion STRING NOT NULL,
        temperatura DOUBLE, 
        humedad_relativa DOUBLE,
        velocidad_viento DOUBLE,
        direccion_viento DOUBLE,
        precipitacon DOUBLE,
        presion DOUBLE,
        radiacion_solar DOUBLE
       
    )
    USING {FORMATO}
    {ubicacion("fact_meteo")}
    """)

26/10/24 21:04:42 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/fact_meteo was not found. Was it deleted very recently?


DataFrame[]

In [14]:
# fact_trafico
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_trafico")}(
        id_tiempo INT NOT NULL,
        id_espira STRING NOT NULL,
        intensidad DOUBLE
       
    )
    USING {FORMATO}
    {ubicacion("fact_trafico")}
    """)

26/10/24 21:05:43 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/fact_trafico was not found. Was it deleted very recently?


DataFrame[]

In [15]:
# fact_prediccion
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_prediccion")}(
        modelo STRING NOT NULL, -- gradient_boosting, calendario, tendencia
        indicador STRING NOT NULL, -- no2, trafico, pm10, ruido
        horizonte STRING, -- 24h, mes, anio, 2030
        granularidad STRING,
        fecha_ejecucion DATE,
        fecha_prevista DATE NOT NULL,
        id_tiempo INT, -- solo en predicciones horarias
        id_estacion STRING, -- vacio si es toda la ciudad
        valor_previsto DOUBLE,
        valor_bajo DOUBLE,
        valor_alto DOUBLE       
    )
    USING {FORMATO}
    {ubicacion("fact_prediccion")}
    """)

26/10/24 21:13:18 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/fact_prediccion was not found. Was it deleted very recently?


DataFrame[]

Agregar las tablas de verde y de ruido cuando las tengamos limpias

## 5. Comprobacion

In [17]:
spark.sql(f"SHOW TABLES IN {ESQUEMA}").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|     gold|dim_contaminante|      false|
|     gold|    dim_estacion|      false|
|     gold|      dim_tiempo|      false|
|     gold|       fact_aire|      false|
|     gold|      fact_meteo|      false|
|     gold| fact_prediccion|      false|
|     gold|    fact_trafico|      false|
+---------+----------------+-----------+

